In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('07_groupby') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 20:36:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/03 20:36:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/03 20:36:22 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
df_green = spark.read.parquet('data/pq/green/2020/*', 'data/pq/green/2021/*')

In [4]:
df_green.createOrReplaceTempView('green')

In [9]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
--ORDER BY 
--    1, 2
""")

In [10]:
df_green_revenue.show()

[Stage 7:===================================================>       (7 + 1) / 8]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-06 07:00:00| 241|            169.16|             6|
|2020-01-19 19:00:00|  41| 457.2400000000003|            43|
|2020-01-23 23:00:00| 129| 304.1300000000001|            22|
|2020-01-28 15:00:00| 129|            208.23|             9|
|2020-01-02 08:00:00|  74|1187.4199999999987|            96|
|2020-01-06 16:00:00| 129|250.42000000000002|            15|
|2020-01-07 10:00:00| 171|             38.98|             2|
|2020-01-09 15:00:00| 260| 242.6800000000001|            19|
|2020-01-16 20:00:00|  82| 424.1900000000001|            27|
|2020-01-21 16:00:00|  33| 917.3599999999994|            38|
|2020-01-08 17:00:00| 140|            313.81|             8|
|2020-01-22 08:00:00| 244| 285.3999999999999|            14|
|2020-01-30 14:00:00|  74| 738.6799999999996|            49|
|2020-01-03 07:00:00|  7

In [12]:
df_green_revenue.write.parquet('data/report/revenue/green', mode='overwrite')

26/03/03 20:42:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [13]:
!ls -lh data/report/revenue/green

total 10304
-rw-r--r--@ 1 mishadavies  staff     0B Mar  3 20:42 _SUCCESS
-rw-r--r--@ 1 mishadavies  staff   622K Mar  3 20:42 part-00000-2de2764f-a27c-48f6-8bef-5fac318fd163-c000.snappy.parquet
-rw-r--r--@ 1 mishadavies  staff   644K Mar  3 20:42 part-00001-2de2764f-a27c-48f6-8bef-5fac318fd163-c000.snappy.parquet
-rw-r--r--@ 1 mishadavies  staff   640K Mar  3 20:42 part-00002-2de2764f-a27c-48f6-8bef-5fac318fd163-c000.snappy.parquet
-rw-r--r--@ 1 mishadavies  staff   624K Mar  3 20:42 part-00003-2de2764f-a27c-48f6-8bef-5fac318fd163-c000.snappy.parquet
-rw-r--r--@ 1 mishadavies  staff   620K Mar  3 20:42 part-00004-2de2764f-a27c-48f6-8bef-5fac318fd163-c000.snappy.parquet
-rw-r--r--@ 1 mishadavies  staff   617K Mar  3 20:42 part-00005-2de2764f-a27c-48f6-8bef-5fac318fd163-c000.snappy.parquet
-rw-r--r--@ 1 mishadavies  staff   623K Mar  3 20:42 part-00006-2de2764f-a27c-48f6-8bef-5fac318fd163-c000.snappy.parquet
-rw-r--r--@ 1 mishadavies  staff   752K Mar  3 20:42 part-00007-2de2764f-a27c-4

In [14]:
# Now repeat for the yellow data:
df_yellow = spark.read.parquet('data/pq/yellow/2020/*', 'data/pq/yellow/2021/*')

In [15]:
df_yellow.createOrReplaceTempView('yellow')

In [16]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [17]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

26/03/03 21:00:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/03 21:00:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/03 21:00:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [19]:
# Joins
# 'how='outer'': Join type (outer join)
df_join = df_green_revenue.join(df_yellow_revenue, on=['hour', 'zone'], how='outer')

In [20]:
df_join.show()

[Stage 21:==========================================>              (9 + 3) / 12]

+-------------------+----+------------------+--------------+------------------+--------------+
|               hour|zone|            amount|number_records|            amount|number_records|
+-------------------+----+------------------+--------------+------------------+--------------+
|2020-01-01 00:00:00|   4|              NULL|          NULL|1004.3000000000001|            57|
|2020-01-01 00:00:00|  10|              NULL|          NULL|             42.41|             2|
|2020-01-01 00:00:00|  56|             99.69|             3|              18.1|             2|
|2020-01-01 00:00:00|  61|            526.71|            17|            146.64|             3|
|2020-01-01 00:00:00|  65|199.49000000000004|            10|            409.35|            19|
|2020-01-01 00:00:00|  74|317.09000000000015|            24| 586.2100000000002|            47|
|2020-01-01 00:00:00|  88|              NULL|          NULL| 823.8000000000002|            36|
|2020-01-01 00:00:00| 114|              NULL|     

In [21]:
# Issue: We can't tell which columns are yellow or green
# Rename columns:

df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [23]:
# Now rerun join:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [27]:
df_join.show()
# Note: This has to recalculate every time you run it...

[Stage 35:===================================================>    (11 + 1) / 12]

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|   4|              NULL|                NULL|1004.3000000000001|                   57|
|2020-01-01 00:00:00|  10|              NULL|                NULL|             42.41|                    2|
|2020-01-01 00:00:00|  56|             99.69|                   3|              18.1|                    2|
|2020-01-01 00:00:00|  61|            526.71|                  17|            146.64|                    3|
|2020-01-01 00:00:00|  65|199.49000000000004|                  10|            409.35|                   19|
|2020-01-01 00:00:00|  74|317.09000000000015|                  24| 586.2100000000002|                   47|
|2020-01-01 00:00:00|  88|  

In [28]:
# Now instead of computing everything on the fly, use materialized results that we saved previously:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [29]:
# Now rerun everything above:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [30]:
# And rejoin:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [32]:
# And write the results to a parquet:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

26/03/03 21:15:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [33]:
# Now read the parquet file:
df_join = spark.read.parquet('data/report/revenue/total')

In [34]:
# Show the df_join
df_join

DataFrame[hour: timestamp, zone: int, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint]

In [35]:
# Now we're going to get a df of a different file which is smaller
df_zones = spark.read.parquet('zones/')

In [36]:
# Now join based on their common name (zone = LocationID)
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [37]:
df_result.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|LocationID|  Borough|                Zone|service_zone|
+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|2020-01-01 00:00:00|  12|              NULL|                NULL|             107.0|                    6|        12|Manhattan|        Battery Park| Yellow Zone|
|2020-01-01 00:00:00|  15|              NULL|                NULL|             34.09|                    1|        15|   Queens|Bay Terrace/Fort ...|   Boro Zone|
|2020-01-01 00:00:00|  17|195.03000000000003|                   9|220.20999999999998|                    8|        17| Brooklyn|             Bedford|   Boro Zone|
|2020-01-01 00:00:00| 

In [38]:
# Save as a parquet and drop the columns LocationID and zone
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')

26/03/03 21:19:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [39]:
df_result.show()
# Note: You won't see a change until there's an executable action on tmp/revenue-zones

+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|LocationID|  Borough|                Zone|service_zone|
+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|2020-01-01 00:00:00|  12|              NULL|                NULL|             107.0|                    6|        12|Manhattan|        Battery Park| Yellow Zone|
|2020-01-01 00:00:00|  15|              NULL|                NULL|             34.09|                    1|        15|   Queens|Bay Terrace/Fort ...|   Boro Zone|
|2020-01-01 00:00:00|  17|195.03000000000003|                   9|220.20999999999998|                    8|        17| Brooklyn|             Bedford|   Boro Zone|
|2020-01-01 00:00:00| 